# 01b. 채널 + 영상 정보 수집 — `playlistItems.list` + `videos.list` API

**목적:** 01a 에서 얻은 `uploads_playlist` 를 이용해  
각 채널의 최근 영상 ID 목록과 영상별 상세 지표를 모두 수집한다.

| 표시 | 의미 |
|------|------|
| ⭐ MUST | 이탈 예측 피처 생성에 필수 |
| ○ OPTIONAL | 있으면 좋지만 없어도 됨 |

**API Quota 비용 (채널 1개 기준)**

| 호출 | 비용 |
|------|------|
| `channels.list` (uploads playlist 획득) | ~0.02 unit |
| `playlistItems.list` 1회 (50개) | 1 unit |
| `videos.list` 1회 (50개 배치) | 1 unit |
| **채널당 합계** | **~2.02 unit** |

## 0. 설정

In [ ]:
import os
import json
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY')
assert API_KEY, '.env 파일에 YOUTUBE_API_KEY 를 설정하세요'

youtube = build('youtube', 'v3', developerKey=API_KEY)

ROOT     = Path('../../')
CSV_PATH = ROOT / 'data' / 'raw' / 'youtube_channels.csv'
OUT_DIR  = ROOT / 'data' / 'raw' / 'videos'
JSON_DIR = OUT_DIR / 'json'        # 채널별 원시 API 응답
CSV_DIR  = OUT_DIR / 'csv'         # MUST 필드 가공본
JSON_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

MAX_VIDEOS_PER_CHANNEL = 50   # 채널당 최근 영상 수 (config.yaml 값과 일치)
REQUEST_DELAY_SEC      = 0.5  # 요청 간격

# ─────────────────────────────────────────────
# 처리할 채널 범위 (youtube_channels.csv 의 행 인덱스, 양 끝 포함)
# 예: 0~49번 채널만 → START_IDX=0, END_IDX=49
# 다음 배치는 START_IDX=50, END_IDX=99 식으로 이어서 실행
# ─────────────────────────────────────────────
START_IDX = 0
END_IDX   = 49

# quota 초과 등으로 중단되면 이 변수에 마지막으로 완료된 인덱스가 저장된다
last_done_idx = None

print('API 연결 완료')
print(f'처리 범위: {START_IDX} ~ {END_IDX} (양 끝 포함)')

## 1. 채널 슬라이스 불러오기 (`START_IDX` ~ `END_IDX`)

원본 CSV 의 행 번호(`idx`)를 그대로 보존하여, JSON · CSV 결과물에서  
`youtube_channels.csv` 의 어느 행에 해당하는지 한 눈에 매칭 가능하게 한다.

In [ ]:
df_all = pd.read_csv(CSV_PATH)

# 원본 CSV 의 행 인덱스를 'idx' 컬럼으로 보존 (JSON·결과 CSV 에서 매칭용)
df_all = df_all.reset_index().rename(columns={'index': 'idx'})

# END_IDX 가 마지막 행을 넘어가도 안전하게 클램프
end_clamped = min(END_IDX, len(df_all) - 1)
df = df_all.iloc[START_IDX:end_clamped + 1].copy()

print(f'전체 채널 수: {len(df_all):,}')
print(f'이번 배치 범위: idx {START_IDX} ~ {end_clamped} ({len(df)}개 채널)')
df[['idx', 'channel_name', 'youtube_channel_id']].head()

## 2. `channels.list` — uploads 플레이리스트 ID 확보

`playlistItems.list` 를 호출하려면 채널의 **uploads playlist ID** 가 필요하다.  
01a 에서 저장한 CSV 가 있으면 재사용, 없으면 여기서 직접 호출한다.

In [ ]:
must_csv = ROOT / 'data' / 'raw' / 'channels' / 'csv' / 'sample_channels_must.csv'

if must_csv.exists():
    df_ch_all = pd.read_csv(must_csv)
    print('01a 에서 저장된 채널 정보 사용')
else:
    # 01a 를 먼저 실행하지 않은 경우 직접 호출
    channel_ids = df['youtube_channel_id'].tolist()
    resp = youtube.channels().list(
        part='contentDetails,statistics,snippet',
        id=','.join(channel_ids)
    ).execute()
    rows = []
    for item in resp['items']:
        s  = item.get('snippet', {})
        cd = item.get('contentDetails', {}).get('relatedPlaylists', {})
        st = item.get('statistics', {})
        rows.append({
            'channel_id':       item['id'],
            'title':            s.get('title'),
            'published_at':     s.get('publishedAt'),
            'uploads_playlist': cd.get('uploads'),
            'subscriber_count': st.get('subscriberCount'),
            'video_count':      st.get('videoCount'),
        })
    df_ch_all = pd.DataFrame(rows)
    print('channels.list 직접 호출 완료')

# 이번 배치에 해당하는 채널만 추리고, 원본 idx 를 부여
# (df_ch_all 의 행 순서는 channels.list 응답 순서라 df 와 다를 수 있음)
df_ch = (
    df[['idx', 'youtube_channel_id']]
        .rename(columns={'youtube_channel_id': 'channel_id'})
        .merge(df_ch_all, on='channel_id', how='left')
        .sort_values('idx')
        .reset_index(drop=True)
)

print(f'이번 배치 채널 정보: {len(df_ch)}개')
df_ch[['idx', 'channel_id', 'title', 'uploads_playlist']].head()

## 3. 채널별 수집 루프 — `playlistItems.list` + `videos.list` 통합

채널 한 개씩 처리하면서:

1. `playlistItems.list` 로 최근 영상 ID·업로드 일자 수집
2. `videos.list` 로 영상별 상세 지표 수집
3. 채널 단위로 즉시 JSON 저장 (`json/{idx:04d}_{channel_id}.json`)
4. **`quotaExceeded` 발생 시 루프를 즉시 중단** → 그 시점까지의 데이터로 CSV 저장 (Cell 30)

| 필드 | 위치 | 중요도 |
|------|------|--------|
| `videoId` | contentDetails | ⭐ MUST |
| `publishedAt` | snippet | ⭐ MUST (업로드 타임스탬프) |
| `title` | snippet | ○ |
| `position` | snippet | ○ 재생목록 내 순서 |

In [ ]:
VIDEO_PARTS = (
    'snippet,contentDetails,statistics,status,'
    'topicDetails,recordingDetails,localizations'
)


def get_playlist_videos(playlist_id, max_results=50):
    """uploads 플레이리스트에서 최근 영상 ID 목록과 업로드 날짜 반환"""
    items = []
    next_page = None
    while len(items) < max_results:
        kwargs = dict(
            part='snippet,contentDetails,status',
            playlistId=playlist_id,
            maxResults=min(50, max_results - len(items))
        )
        if next_page:
            kwargs['pageToken'] = next_page
        resp = youtube.playlistItems().list(**kwargs).execute()
        items.extend(resp.get('items', []))
        next_page = resp.get('nextPageToken')
        if not next_page:
            break
        time.sleep(REQUEST_DELAY_SEC)
    return items


def get_video_details(video_ids):
    """video_ids 리스트를 50개씩 배치 호출하여 상세 정보 반환"""
    all_items = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        resp = youtube.videos().list(
            part=VIDEO_PARTS,
            id=','.join(batch)
        ).execute()
        all_items.extend(resp.get('items', []))
        time.sleep(REQUEST_DELAY_SEC)
    return all_items


def _err_reason(e: HttpError) -> str:
    try:
        return e.error_details[0]['reason']
    except Exception:
        return f'HTTP {e.resp.status}'


def is_quota_error(e: HttpError) -> bool:
    """API 토큰(quota) 소진 여부 — 발생 시 즉시 중단해야 함"""
    reason = _err_reason(e)
    return reason in ('quotaExceeded', 'rateLimitExceeded', 'dailyLimitExceeded')


# 채널별 수집 결과 (channel_id → raw items)
playlist_data = {}    # playlistItems 응답
video_data    = {}    # videos.list 응답
idx_map       = {}    # channel_id → 원본 CSV idx
skipped       = []    # (idx, channel_id, reason)
last_done_idx = None  # 완전히 처리(JSON 저장)된 마지막 idx
quota_hit     = False

for _, row in df_ch.iterrows():
    cid         = row['channel_id']
    idx         = int(row['idx'])
    title       = row.get('title', cid)
    playlist_id = row.get('uploads_playlist')

    idx_map[cid] = idx

    if not playlist_id or pd.isna(playlist_id):
        print(f'  [{idx:>4}] SKIP {title} — uploads_playlist 없음')
        skipped.append((idx, cid, 'no_playlist'))
        last_done_idx = idx  # 의도적으로 건너뛴 것도 "처리 완료"로 간주
        continue

    print(f'  [{idx:>4}] 수집 중: {title}')

    # 1) playlistItems.list
    try:
        items = get_playlist_videos(playlist_id, MAX_VIDEOS_PER_CHANNEL)
        playlist_data[cid] = items
        print(f'        playlistItems → {len(items)}개')
    except HttpError as e:
        reason = _err_reason(e)
        if is_quota_error(e):
            print(f'        ✗ QUOTA 소진 (playlistItems) — {reason} → 루프 중단')
            quota_hit = True
            break
        print(f'        ✗ SKIP — {reason}')
        skipped.append((idx, cid, reason))
        last_done_idx = idx
        time.sleep(REQUEST_DELAY_SEC)
        continue

    # 2) videos.list
    video_ids = [
        it['contentDetails']['videoId']
        for it in items
        if it.get('contentDetails', {}).get('videoId')
    ]
    if video_ids:
        try:
            video_data[cid] = get_video_details(video_ids)
            print(f'        videos.list  → {len(video_data[cid])}개')
        except HttpError as e:
            reason = _err_reason(e)
            if is_quota_error(e):
                print(f'        ✗ QUOTA 소진 (videos.list) — {reason} → 루프 중단')
                # playlistItems 는 받았지만 videos 는 못 받음 → 이 채널은 미완료로 간주
                playlist_data.pop(cid, None)
                quota_hit = True
                break
            print(f'        ✗ videos.list SKIP — {reason}')
            skipped.append((idx, cid, f'videos:{reason}'))

    # 3) 채널 단위 즉시 JSON 저장 (idx 를 파일명·payload 양쪽에)
    json_path = JSON_DIR / f'{idx:04d}_{cid}.json'
    payload = {
        'idx':            idx,
        'channel_id':     cid,
        'channel_title':  title,
        'playlist_items': playlist_data.get(cid, []),
        'video_details':  video_data.get(cid, []),
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    last_done_idx = idx
    time.sleep(REQUEST_DELAY_SEC)

print('\n' + '─' * 60)
print(f'수집 종료 — 성공 {len(video_data)}개 / 스킵 {len(skipped)}개')
print(f'마지막 완료 idx: {last_done_idx}')
if quota_hit:
    print(f'⚠ API quota 소진으로 중단됨')
    print(f'  → 다음 실행 시 START_IDX = {(last_done_idx + 1) if last_done_idx is not None else START_IDX}')
else:
    print(f'  → 다음 실행 시 START_IDX = {(last_done_idx + 1) if last_done_idx is not None else START_IDX}')
if skipped:
    print(f'스킵 채널 일부: {skipped[:5]}{" ..." if len(skipped) > 5 else ""}')

In [ ]:
# playlistItems 전체 필드 확인 (첫 번째 채널의 첫 번째 영상)
# 영상이 한 건도 없으면 메시지만 출력하고 넘어감
_channels_with_items = [cid for cid, items in playlist_data.items() if items]
if not _channels_with_items:
    print('표시할 playlistItems 가 없습니다 (수집된 영상 0개)')
else:
    first_channel_id = _channels_with_items[0]
    first_item = playlist_data[first_channel_id][0]
    print(f'=== playlistItems 전체 응답 구조 (channel: {first_channel_id}) ===')
    print(json.dumps(first_item, indent=2, ensure_ascii=False))

In [ ]:
# MUST 필드 추출 (idx 포함)
playlist_rows = []
for channel_id, items in playlist_data.items():
    for item in items:
        s  = item.get('snippet', {})
        cd = item.get('contentDetails', {})
        st = item.get('status', {})
        playlist_rows.append({
            'idx':           idx_map.get(channel_id),  # 원본 CSV 행 번호
            'channel_id':    channel_id,
            'video_id':      cd.get('videoId'),           # ⭐ MUST  videos.list 에 사용
            'published_at':  s.get('publishedAt'),         # ⭐ MUST  업로드 날짜·시간
            'title':         s.get('title'),               # ○
            'position':      s.get('position'),            # ○ 재생목록 순서
            'privacy':       st.get('privacyStatus'),      # ○ public / unlisted / private
        })

# 빈 케이스에도 컬럼이 살아있도록 명시
df_playlist = pd.DataFrame(
    playlist_rows,
    columns=['idx', 'channel_id', 'video_id', 'published_at', 'title', 'position', 'privacy'],
)
print(f'총 영상 행 수: {len(df_playlist)}')
df_playlist.head(10)

## 4. `videos.list` 응답 구조 살펴보기

`videos.list` 호출은 이미 Cell 8 의 통합 루프에서 채널별로 수행되어  
결과가 `video_data` 에 담겨 있다. 아래 셀은 그 응답에서 어떤 필드를 뽑아 쓸지를 확인하는 용도다.

| Part | 주요 필드 | 중요도 |
|------|-----------|--------|
| `snippet` | publishedAt, title, tags, categoryId, defaultLanguage | ⭐/○ |
| `contentDetails` | duration, definition, caption, contentRating | ⭐/○ |
| `statistics` | viewCount, likeCount, commentCount | ⭐ MUST |
| `status` | privacyStatus, madeForKids, embeddable | ○ |
| `topicDetails` | topicCategories | ○ |
| `recordingDetails` | recordingDate | ○ |
| `localizations` | 언어별 제목/설명 | ○ |

In [ ]:
# videos.list 호출은 Cell 8 의 통합 루프에서 이미 수행됨 — 여기서는 결과만 확인
print(f'video_data 채널 수: {len(video_data)}')
total_videos = sum(len(v) for v in video_data.values())
print(f'총 영상 응답 수: {total_videos}')

In [ ]:
# 전체 응답 구조 확인 (첫 번째 채널, 첫 번째 영상)
_channels_with_videos = [cid for cid, vids in video_data.items() if vids]
if not _channels_with_videos:
    print('표시할 videos.list 응답이 없습니다 (수집된 영상 0개)')
else:
    first_channel_id = _channels_with_videos[0]
    first_video = video_data[first_channel_id][0]
    print(f'=== videos.list 전체 응답 구조 (channel: {first_channel_id}) ===')
    print(json.dumps(first_video, indent=2, ensure_ascii=False))

### 4-1. snippet

In [9]:
rows = []
for channel_id, videos in video_data.items():
    for v in videos:
        s = v.get('snippet', {})
        rows.append({
            'channel_id':             channel_id,
            'video_id':               v['id'],
            'published_at':           s.get('publishedAt'),           # ⭐ MUST  업로드 타임스탬프
            'title':                  s.get('title'),                 # ○
            'category_id':            s.get('categoryId'),            # ○ YouTube 영상 카테고리
            'tags':                   s.get('tags'),                  # ○ 태그 목록
            'default_language':       s.get('defaultLanguage'),       # ○
            'default_audio_language': s.get('defaultAudioLanguage'),  # ○
            'live_broadcast_content': s.get('liveBroadcastContent'),  # ○ none/live/upcoming
        })

pd.DataFrame(rows).head(10)

,channel_id,video_id,published_at,title,category_id,tags,default_language,default_audio_language,live_broadcast_content
0,UCGPHpVMxzO2rc5SqhGIc2tg,0ms82oCpSXY,2017-03-08T06:56:28Z,김정우의원 / 군포시 보훈회관 건립에 관한 지방재정 투자심사 통과,22,None,ko,ko,none
1,UCyABUa7lzjsV4o5PfSFqymw,Lq9R7Susaeg,2017-11-11T12:03:32Z,픽이 가관인 심해 속 망치를 든 스타트!오늘부로 라인하르트 입덕합니다!!!!,22,None,ko,ko,none
2,UCyABUa7lzjsV4o5PfSFqymw,XwcIz07B_RM,2017-10-09T06:47:10Z,[start]:심해 매드무비-둠피스트,22,None,ko,en,none
3,UCyABUa7lzjsV4o5PfSFqymw,M7vYRNXEkwk,2017-08-19T12:12:26Z,"여기서 만큼은 심해가 아니다!중딩 스타트의 플레 루시우볼!(feat,뽈쟁이,버틀너버",22,None,ko,ko,none
4,UCo3Yj54VtkEvQX9cLHKklzw,VhXviHrA_yw,2019-12-11T05:58:53Z,"191211 언제나 국민의편, 국회의원 장정숙",22,None,ko,ko,none
5,UCo3Yj54VtkEvQX9cLHKklzw,KoovsJOVOEg,2018-07-03T04:16:52Z,[국회의원 장정숙] 전반기 교육문화체육관광위원회 활동,22,"[국회의원, 장정숙, 교육문화체육관광위원회, 교문위, 문체부, 문화계 블랙리스트, ...",ko,ko,none
6,UCo3Yj54VtkEvQX9cLHKklzw,eM25H_bOkJ4,2018-05-28T02:16:23Z,171024 [장정숙 의원] 2017 국정감사_국립대 신입간호사 임금착취 문제 해결,22,"[국립대병원, 전남대병원, 임금체불, 노동착취, 신입간호사, 국정감사, 장정숙, 국...",ko,ko,none


### 4-2. contentDetails — ⭐ duration (영상 길이 → Shorts 판별)

In [ ]:
import re

def parse_duration_sec(iso_duration):
    """ISO 8601 기간 문자열 → 초 (예: PT1M30S → 90)"""
    if not iso_duration:
        return None
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    m = re.match(pattern, iso_duration)
    if not m:
        return None
    h, mn, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mn * 60 + s

rows = []
for channel_id, videos in video_data.items():
    for v in videos:
        cd = v.get('contentDetails', {})
        duration_raw = cd.get('duration')
        duration_sec = parse_duration_sec(duration_raw)
        rows.append({
            'channel_id':       channel_id,
            'video_id':         v['id'],
            'duration_raw':     duration_raw,                         # ISO 8601
            'duration_sec':     duration_sec,                         # ⭐ MUST  영상 길이(초)
            'is_shorts':        duration_sec is not None and duration_sec <= 60,  # ⭐ MUST  Shorts 여부
            'definition':       cd.get('definition'),                 # ○ hd / sd
            'caption':          cd.get('caption'),                    # ○ 자막 여부
            'licensed_content': cd.get('licensedContent'),            # ○
            'content_rating':   cd.get('contentRating'),              # ○ 연령 제한 등
            'projection':       cd.get('projection'),                 # ○ rectangular / 360
        })

df_content = pd.DataFrame(
    rows,
    columns=['channel_id', 'video_id', 'duration_raw', 'duration_sec',
             'is_shorts', 'definition', 'caption', 'licensed_content',
             'content_rating', 'projection'],
)
if len(df_content):
    print(f'Shorts 비율: {df_content["is_shorts"].mean():.1%}')
else:
    print('Shorts 비율: — (영상 0개)')
df_content.head(10)

### 4-3. statistics — ⭐ 조회수·좋아요·댓글 (핵심 피처)

In [ ]:
rows = []
for channel_id, videos in video_data.items():
    for v in videos:
        st = v.get('statistics', {})
        rows.append({
            'channel_id':    channel_id,
            'video_id':      v['id'],
            'view_count':    st.get('viewCount'),      # ⭐ MUST  조회수
            'like_count':    st.get('likeCount'),       # ⭐ MUST  좋아요 수
            'comment_count': st.get('commentCount'),   # ⭐ MUST  댓글 수
            'favorite_count': st.get('favoriteCount'), # ○ (항상 0, YouTube 정책 변경)
        })

df_stats = pd.DataFrame(
    rows,
    columns=['channel_id', 'video_id', 'view_count', 'like_count', 'comment_count', 'favorite_count'],
)
# 숫자 타입 변환
for col in ['view_count', 'like_count', 'comment_count']:
    df_stats[col] = pd.to_numeric(df_stats[col], errors='coerce')

if len(df_stats):
    print(df_stats[['view_count', 'like_count', 'comment_count']].describe())
else:
    print('통계 요약: — (영상 0개)')
df_stats.head(10)

### 4-4. status

In [12]:
rows = []
for channel_id, videos in video_data.items():
    for v in videos:
        s = v.get('status', {})
        rows.append({
            'channel_id':    channel_id,
            'video_id':      v['id'],
            'privacy_status': s.get('privacyStatus'),                 # ○ public/unlisted/private
            'upload_status':  s.get('uploadStatus'),                  # ○ processed/uploaded
            'embeddable':     s.get('embeddable'),                    # ○
            'made_for_kids':  s.get('madeForKids'),                   # ○ 어린이용 여부
            'public_stats_viewable': s.get('publicStatsViewable'),    # ○
        })

pd.DataFrame(rows).head(10)

,channel_id,video_id,privacy_status,upload_status,embeddable,made_for_kids,public_stats_viewable
0,UCGPHpVMxzO2rc5SqhGIc2tg,0ms82oCpSXY,public,processed,True,False,True
1,UCyABUa7lzjsV4o5PfSFqymw,Lq9R7Susaeg,public,processed,True,False,True
2,UCyABUa7lzjsV4o5PfSFqymw,XwcIz07B_RM,public,processed,True,False,True
3,UCyABUa7lzjsV4o5PfSFqymw,M7vYRNXEkwk,public,processed,True,False,True
4,UCo3Yj54VtkEvQX9cLHKklzw,VhXviHrA_yw,public,processed,True,False,True
5,UCo3Yj54VtkEvQX9cLHKklzw,KoovsJOVOEg,public,processed,True,False,True
6,UCo3Yj54VtkEvQX9cLHKklzw,eM25H_bOkJ4,public,processed,True,False,True


### 4-5. topicDetails

In [13]:
rows = []
for channel_id, videos in video_data.items():
    for v in videos:
        td = v.get('topicDetails', {})
        rows.append({
            'channel_id':       channel_id,
            'video_id':         v['id'],
            'topic_categories': td.get('topicCategories'),   # ○ Wikipedia 카테고리
            'relevant_topic_ids': td.get('relevantTopicIds'), # ○
        })

pd.DataFrame(rows).head(5)

,channel_id,video_id,topic_categories,relevant_topic_ids
0,UCGPHpVMxzO2rc5SqhGIc2tg,0ms82oCpSXY,"[https://en.wikipedia.org/wiki/Politics, https...",None
1,UCyABUa7lzjsV4o5PfSFqymw,Lq9R7Susaeg,"[https://en.wikipedia.org/wiki/Action_game, ht...",None
2,UCyABUa7lzjsV4o5PfSFqymw,XwcIz07B_RM,"[https://en.wikipedia.org/wiki/Action_game, ht...",None
3,UCyABUa7lzjsV4o5PfSFqymw,M7vYRNXEkwk,[https://en.wikipedia.org/wiki/Strategy_video_...,None
4,UCo3Yj54VtkEvQX9cLHKklzw,VhXviHrA_yw,"[https://en.wikipedia.org/wiki/Politics, https...",None


### 4-6. recordingDetails

In [14]:
rows = []
for channel_id, videos in video_data.items():
    for v in videos:
        rd = v.get('recordingDetails', {})
        rows.append({
            'channel_id':    channel_id,
            'video_id':      v['id'],
            'recording_date': rd.get('recordingDate'),  # ○ 촬영 날짜 (보통 None)
        })

pd.DataFrame(rows).head(5)

,channel_id,video_id,recording_date
0,UCGPHpVMxzO2rc5SqhGIc2tg,0ms82oCpSXY,None
1,UCyABUa7lzjsV4o5PfSFqymw,Lq9R7Susaeg,None
2,UCyABUa7lzjsV4o5PfSFqymw,XwcIz07B_RM,None
3,UCyABUa7lzjsV4o5PfSFqymw,M7vYRNXEkwk,None
4,UCo3Yj54VtkEvQX9cLHKklzw,VhXviHrA_yw,None


## 5. 이탈 예측 피처 생성 프리뷰

수집한 데이터로 만들 수 있는 피처 카테고리별 예시

In [15]:
# 채널별 업로드 타임스탬프 정렬
df_pl = df_playlist.copy()
df_pl['published_at'] = pd.to_datetime(df_pl['published_at'])
df_pl = df_pl.sort_values(['channel_id', 'published_at'])

feature_rows = []
for cid, grp in df_pl.groupby('channel_id'):
    timestamps = grp['published_at'].sort_values()
    intervals  = timestamps.diff().dt.total_seconds() / 86400  # 일 단위

    # A. 업로드 활동성
    avg_interval = intervals.mean()        # ⭐ 평균 업로드 간격(일)
    std_interval = intervals.std()         # ⭐ 주기 표준편차
    max_gap      = intervals.max()         # ⭐ 최대 공백 기간

    n = len(timestamps)
    recent_half = timestamps.tail(n // 2 if n > 1 else 1)
    older_half  = timestamps.head(n // 2 if n > 1 else 1)

    # 기간별 업로드 수 변화
    freq_recent = len(recent_half)
    freq_older  = len(older_half)
    freq_change = (freq_recent - freq_older) / max(freq_older, 1)

    feature_rows.append({
        'channel_id':      cid,
        'video_count':     n,
        'avg_interval_d':  round(avg_interval, 1),
        'std_interval_d':  round(std_interval, 1),
        'max_gap_d':       round(max_gap, 1),
        'freq_change_rate': round(freq_change, 3),
    })

df_act = pd.DataFrame(feature_rows)
print('A카테고리 업로드 활동성 피처 프리뷰')
df_act

A카테고리 업로드 활동성 피처 프리뷰


,channel_id,video_count,avg_interval_d,std_interval_d,max_gap_d,freq_change_rate
0,UCGPHpVMxzO2rc5SqhGIc2tg,1,NaN,NaN,NaN,0.0
1,UCo3Yj54VtkEvQX9cLHKklzw,3,281.1,346.5,526.1,0.0
2,UCyABUa7lzjsV4o5PfSFqymw,3,42.0,12.4,50.8,0.0


In [16]:
# B카테고리 — 성과 지표 (statistics 기반)
import math

def _round(x, ndigits=0):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    return round(x, ndigits) if ndigits else round(x)

df_s = df_stats.copy()
perf_rows = []
for cid, grp in df_s.groupby('channel_id'):
    n = len(grp)
    views  = grp['view_count']
    likes  = grp['like_count']
    comms  = grp['comment_count']

    recent_n = max(1, n // 2)
    view_trend_ratio = views.tail(recent_n).mean() / max(views.mean(), 1)
    engagement_rate  = (likes + comms).mean() / max(views.mean(), 1)

    perf_rows.append({
        'channel_id':       cid,
        'video_count':      n,
        'avg_view':         _round(views.mean()),               # ⭐
        'view_std':         _round(views.std()),                 # ⭐ (영상 1개면 NaN)
        'view_trend_ratio': _round(view_trend_ratio, 3),         # ⭐ 최근 조회수 비율
        'avg_like':         _round(likes.mean()),                # ⭐
        'avg_comment':      _round(comms.mean()),                # ⭐
        'engagement_rate':  _round(engagement_rate, 5),          # ⭐
    })

df_perf = pd.DataFrame(perf_rows)
print('B카테고리 성과 지표 피처 프리뷰')
df_perf

B카테고리 성과 지표 피처 프리뷰


,channel_id,video_count,avg_view,view_std,view_trend_ratio,avg_like,avg_comment,engagement_rate
0,UCGPHpVMxzO2rc5SqhGIc2tg,1,41,NaN,1.000,2,0,0.04878
1,UCo3Yj54VtkEvQX9cLHKklzw,3,164,79.0,0.456,4,0,0.02434
2,UCyABUa7lzjsV4o5PfSFqymw,3,314,38.0,0.882,27,3,0.09554


## 6. 결과 저장

In [ ]:
# Cell 8 의 루프가 채널 단위로 이미 JSON 을 저장했으므로,
# 여기서는 MUST 필드 통합 CSV 만 만들면 된다.

if last_done_idx is None:
    print('처리된 채널이 없습니다 — CSV 저장 건너뜀')
    print(f'다음 실행 시 START_IDX = {START_IDX} 그대로 재시도')
else:
    # 처리된 idx 범위 (파일명용) — START_IDX 부터 실제 마지막 완료 idx 까지
    range_start = START_IDX
    range_end   = last_done_idx
    suffix      = '_partial' if quota_hit else ''
    fname       = f'videos_{range_start:04d}-{range_end:04d}{suffix}.csv'

    # MUST 필드 통합 CSV (idx + channel_title + video_title 포함)
    df_must_stats = df_stats.copy()
    df_must_stats[['view_count', 'like_count', 'comment_count']] = (
        df_must_stats[['view_count', 'like_count', 'comment_count']]
            .apply(pd.to_numeric, errors='coerce')
    )

    df_combined = (
        df_playlist[['idx', 'channel_id', 'video_id', 'title', 'published_at']]
            .rename(columns={'title': 'video_title'})
            .merge(df_ch[['channel_id', 'title']].rename(columns={'title': 'channel_title'}),
                   on='channel_id', how='left')
            .merge(df_content[['video_id', 'duration_sec', 'is_shorts']],
                   on='video_id', how='left')
            .merge(df_must_stats[['video_id', 'view_count', 'like_count', 'comment_count']],
                   on='video_id', how='left')
    )

    # 컬럼 순서 정리: idx → id → 이름 → 메타 → 지표
    df_combined = df_combined[[
        'idx',
        'channel_id', 'channel_title',
        'video_id', 'video_title',
        'published_at', 'duration_sec', 'is_shorts',
        'view_count', 'like_count', 'comment_count',
    ]].sort_values(['idx', 'published_at']).reset_index(drop=True)

    out_csv = CSV_DIR / fname
    df_combined.to_csv(out_csv, index=False, encoding='utf-8-sig')

    n_unique_idx = df_combined['idx'].nunique() if len(df_combined) else 0
    print('=' * 60)
    print(f'MUST 필드 통합 CSV 저장: {out_csv}')
    print(f'  처리 범위(idx):  {range_start} ~ {range_end}')
    print(f'  영상 행 수:      {len(df_combined)}')
    print(f'  영상 수집 채널:  {n_unique_idx} / 시도 {len(df_ch)}')
    if quota_hit:
        print(f'  ⚠ partial — quota 소진으로 중단됨')
    print()
    print(f'다음 배치 실행 시 Cell 2 에서:')
    print(f'  START_IDX = {range_end + 1}')
    print(f'  END_IDX   = {range_end + 50}   # 예: 50개씩')
    print('=' * 60)
    df_combined.head()

## 7. 수집 필드 최종 정리

### ⭐ MUST — 이탈 예측 피처에 직접 사용

| 필드 | 출처 | 피처 카테고리 |
|------|------|---------------|
| `published_at` (playlistItems) | playlistItems.snippet | A. 업로드 활동성 |
| `duration_sec` | videos.contentDetails | A. Shorts 비율, 영상 길이 |
| `view_count` | videos.statistics | B. 조회수 추세·평균 |
| `like_count` | videos.statistics | B. 참여율 |
| `comment_count` | videos.statistics | B/C. 참여율·댓글 활동 |

### ○ OPTIONAL — 추가 분석 또는 DL 팀 활용

| 필드 | 출처 | 활용 아이디어 |
|------|------|---------------|
| `tags` | videos.snippet | 콘텐츠 다양성 피처 |
| `category_id` | videos.snippet | 카테고리 인코딩 |
| `caption` | videos.contentDetails | 자막 여부 피처 |
| `definition` | videos.contentDetails | 영상 품질 피처 |
| `topic_categories` | videos.topicDetails | 토픽 분류 |
| `privacy_status` | videos.status | 비공개 전환 감지 |
| `made_for_kids` | videos.status | 채널 특성 |

### 다음 단계: 01c 노트북 (선택)
DL 팀이 감성 분석을 할 경우 `commentThreads.list` 로 댓글 텍스트 수집 필요